In [1]:
import pandas as pd


In [2]:
df = pd.read_csv('pizza_sales.csv')
df.head()

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1,1,hawaiian_m,1,1/1/2015,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2,2,classic_dlx_m,1,1/1/2015,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3,2,five_cheese_l,1,1/1/2015,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4,2,ital_supr_l,1,1/1/2015,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5,2,mexicana_m,1,1/1/2015,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


In [6]:
df.info()
#info function used to check all column data types. Notably, fields order_date and order_time data type needs to be fixed. 


<class 'pandas.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   pizza_id           48620 non-null  int64         
 1   order_id           48620 non-null  int64         
 2   pizza_name_id      48620 non-null  str           
 3   quantity           48620 non-null  int64         
 4   order_date         48620 non-null  datetime64[us]
 5   order_time         48620 non-null  datetime64[us]
 6   unit_price         48620 non-null  float64       
 7   total_price        48620 non-null  float64       
 8   pizza_size         48620 non-null  str           
 9   pizza_category     48620 non-null  str           
 10  pizza_ingredients  48620 non-null  str           
 11  pizza_name         48620 non-null  str           
dtypes: datetime64[us](2), float64(2), int64(3), str(5)
memory usage: 9.6 MB


Converting all date time fields to type dateTime using a function for reusability

In [4]:
def converToDateTime(df,columns):
    for col in columns:
        df[col] = pd.to_datetime(df[col], format ='mixed')

In [5]:
df['order_time'] = df['order_date'].astype('str') + ' ' + df['order_time']
converToDateTime(df,['order_date','order_time'])

Conver to Currency

In [7]:
def currencyConverter(df, prefix, *args):
    for col in args:
        df[col] = df[col].map(f"{prefix}""{:,.2f}".format)

PercentageFormatter

In [8]:
def percentageFormatter(df, postfix= '%', *args):
    for col in args:
        df[col] = df[col].map(f'{{:.2f}}{postfix}'.format)

Sales Analysis Function

In [9]:
def salesAnalysis(dataframe, col_name, rename):
    data= dataframe.groupby(col_name).agg(
        Total_Revenue = ('total_price', 'sum'),
        Total_Pizzas_Sold = ('pizza_id','count'),
        Total_Orders = ('order_id', 'nunique'),
    ).reset_index().rename(
        columns = {col_name : rename}
    )

    data['Average_Order_Value'] = data['Total_Revenue']/ data['Total_Orders']
    data['Average_Pizzas_Per_Order'] = (data['Total_Pizzas_Sold']/ data['Total_Orders']).map("{:.2f}".format)

    currencyConverter(data, 'Rs. ','Total_Revenue', 'Average_Order_Value' )
    return data   

Checking for duplicates

In [10]:
df[df.duplicated(subset = 'pizza_id')].count()

pizza_id             0
order_id             0
pizza_name_id        0
quantity             0
order_date           0
order_time           0
unit_price           0
total_price          0
pizza_size           0
pizza_category       0
pizza_ingredients    0
pizza_name           0
dtype: int64

Check data date ranges

In [11]:
latest_date = df['order_date'].max()
earliest_date = df['order_date'].min()
print(f"{earliest_date} - {latest_date}")

2015-01-01 00:00:00 - 2015-12-31 00:00:00


FEATURE ENGINEERING

In [13]:
df['order_day_name']= df['order_date'].dt.day_name()
df['order_month_number'] = df['order_date'].dt.month
df['order_month_name'] = df['order_date'].dt.month_name()
df['order_hour']= df['order_time'].dt.hour

Key Performance Indicators (KPIs)
-	Total Revenue = Sum of total_price
-	Total Pizzas Sold = Sum of quantity
-	Total Orders = Count of unique order_id
-	Average Order Value (AOV) = Total Revenue ÷ Total Orders
-	Average Pizza per Order = Total Pizzas Sold ÷ Total Orders



In [29]:
total_revenue = df['total_price'].sum()
total_pizzas_sold = df['pizza_id'].count()
total_orders = df['order_id'].nunique()
average_order_value = total_revenue / total_orders
average_pizza_per_order = total_pizzas_sold / total_orders

print(f"Total Revenue: Rs. {total_revenue:,.2f}")
print(f"Total Pizzas Sold: {total_pizzas_sold}")
print(f"Total Orders: {total_orders}")
print(f"Average Order Value: Rs. {average_order_value:,.2f}")
print(f"Average Pizzas per Order: {average_pizza_per_order:.2f}")

Total Revenue: Rs. 817,860.05
Total Pizzas Sold: 48620
Total Orders: 21350
Average Order Value: Rs. 38.31
Average Pizzas per Order: 2.28


ANALYSIS

In [15]:
weekdays = ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']
df['order_day_name'] = pd.Categorical(df['order_day_name'], categories=weekdays)

Sales By Day of the Week

In [16]:
#Sales by day
sales_by_day = salesAnalysis(df, 'order_day_name','Order Day')
sales_by_day.columns = sales_by_day.columns.str.replace('_',' ',regex=False)
sales_by_day[['Order Day', 'Total Revenue']]


,Order Day,Total Revenue
0,Sunday,"Rs. 102,116.45"
1,Monday,"Rs. 110,471.60"
2,Tuesday,"Rs. 115,594.45"
3,Wednesday,"Rs. 116,731.20"
4,Thursday,"Rs. 121,650.30"
5,Friday,"Rs. 129,690.90"
6,Saturday,"Rs. 121,605.15"


Sales by the Hour of the day

In [17]:
earliest_order_time = df['order_time'].dt.hour.min().astype('str') +':00'
latest_order_time = df['order_time'].dt.hour.max().astype('str') + ':00'
print('time range for orders: ', earliest_order_time, '-', latest_order_time)

time range for orders:  9:00 - 23:00


In [18]:
sales_by_the_hour = salesAnalysis(df, 'order_hour', 'Order Hour')
sales_by_the_hour.columns = sales_by_the_hour.columns.str.replace('_',' ')
sales_by_the_hour[['Order Hour','Total Revenue']]

,Order Hour,Total Revenue
0,9,Rs. 83.00
1,10,Rs. 303.65
2,11,"Rs. 44,935.80"
3,12,"Rs. 111,877.90"
4,13,"Rs. 106,065.70"
5,14,"Rs. 59,201.40"
6,15,"Rs. 52,992.30"
7,16,"Rs. 70,055.40"
8,17,"Rs. 86,237.45"
9,18,"Rs. 89,296.85"


Sales by the month


In [19]:
#sales_by_month
sales_by_month = df.groupby(['order_month_number','order_month_name']).agg(
    Total_Revenue = ('total_price', 'sum'),
    Total_Pizzas_Sold = ('pizza_id','count'),
    Total_Orders = ('order_id', 'nunique'),
).reset_index().rename(
    columns = {
        'order_month_name': 'Order Month'
    }    
)[['Order Month','Total_Revenue','Total_Pizzas_Sold','Total_Orders']]

sales_by_month['Average_Order_Value'] = sales_by_month['Total_Revenue'] /sales_by_month['Total_Orders']
sales_by_month['Average_Pizza_Per_Order'] = (sales_by_month['Total_Pizzas_Sold']/ sales_by_month['Total_Orders']).map("{:.2f}".format)


currencyConverter(sales_by_month, 'Rs. ','Total_Revenue', 'Average_Order_Value')
sales_by_month.columns = sales_by_month.columns.str.replace('_',' ',regex=False)

sales_by_month[['Order Month','Total Revenue']]


,Order Month,Total Revenue
0,January,"Rs. 71,620.15"
1,February,"Rs. 64,419.45"
2,March,"Rs. 71,301.40"
3,April,"Rs. 70,312.00"
4,May,"Rs. 67,648.80"
5,June,"Rs. 68,161.45"
6,July,"Rs. 70,880.65"
7,August,"Rs. 69,497.30"
8,September,"Rs. 63,803.70"
9,October,"Rs. 68,152.20"


In [20]:
# % Sales by Pizza Category
sales_by_category = salesAnalysis(df, 'pizza_category', 'Pizza Category' )
sales_by_category['Percentage_Sold'] = (sales_by_category['Total_Pizzas_Sold']/total_pizzas_sold * 100)
percentageFormatter(sales_by_category, '%','Percentage_Sold')
sales_by_category.columns = sales_by_category.columns.str.replace('_',' ',regex=False)


sales_by_category[['Pizza Category','Percentage Sold']]

,Pizza Category,Percentage Sold
0,Chicken,22.24%
1,Classic,29.99%
2,Supreme,24.22%
3,Veggie,23.55%


In [21]:
# % Sales by Pizza Sizes
sales_by_size = salesAnalysis(df, 'pizza_size', 'Pizza Size')
sales_by_size['Percentage_Sold'] = sales_by_size['Total_Pizzas_Sold'] / total_pizzas_sold * 100
percentageFormatter(sales_by_size,'%', 'Percentage_Sold')
sales_by_size.columns = sales_by_size.columns.str.replace('_',' ', regex=False)
sales_by_size[['Pizza Size','Percentage Sold']]

,Pizza Size,Percentage Sold
0,L,38.10%
1,M,31.64%
2,S,29.08%
3,XL,1.12%
4,XXL,0.06%


In [22]:
#Total Pizzas Sold by Category
sales_by_category[['Pizza Category','Total Pizzas Sold']]

,Pizza Category,Total Pizzas Sold
0,Chicken,10815
1,Classic,14579
2,Supreme,11777
3,Veggie,11449


In [23]:
# Top 5 highest selling pizzas 
sales_by_pizza_name = df.groupby('pizza_name').agg(
    Total_Orders = ('pizza_id','count')
).sort_values(by="Total_Orders", ascending=False).rename(
    columns={
        'pizza_name': 'Pizza Name',
        'Total_Orders': 'Total Orders'
    }
)
sales_by_pizza_name.head(5).reset_index()

,pizza_name,Total Orders
0,The Classic Deluxe Pizza,2416
1,The Barbecue Chicken Pizza,2372
2,The Hawaiian Pizza,2370
3,The Pepperoni Pizza,2369
4,The Thai Chicken Pizza,2315


In [24]:
#bottom 5 least selling pizzas
sales_by_pizza_name.tail(5).sort_values(by='Total Orders',ascending=True).reset_index()

,pizza_name,Total Orders
0,The Brie Carre Pizza,480
1,The Mediterranean Pizza,923
2,The Calabrese Pizza,927
3,The Spinach Supreme Pizza,940
4,The Soppressata Pizza,957
